In [35]:
import csv
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Part 01

1. Read data

In [33]:
PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"

data_path = DATA_RAW / "nasdaq_exteral_data.csv" 
sample_path = DATA_RAW / "nasdaq_sample_rows_5000.csv"  

# read data row by row
max_rows = 5000
count = 0
with open(data_path, "r", newline="", encoding="utf-8", errors="ignore") as f_in, \
     open(sample_path, "w", newline="", encoding="utf-8") as f_out:
    
    reader = csv.reader(f_in)
    writer = csv.writer(f_out)
    
    for row in reader:
        writer.writerow(row)
        count += 1
        if count >= max_rows:
            break

# verify
print("Writtern records:", count)
print("sample file:", sample_path, "exists:", sample_path.exists())

# preview news
news_preview = pd.read_csv(sample_path, nrows=5)
print(news_preview.columns)

news = pd.read_csv(sample_path)
news["Stock_symbol"].value_counts().head(10)

Writtern records: 5000
sample file: ../data/raw/nasdaq_sample_rows_5000.csv exists: True
Index(['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url',
       'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary',
       'Textrank_summary', 'Lexrank_summary'],
      dtype='object')


Stock_symbol
AAL     2962
AA      1525
A        379
AADR      77
AACG      49
AAAU       7
Name: count, dtype: int64

2. Construct event list

In [25]:
TARGET_TICKER = "AAL"
news["Date"] = pd.to_datetime(news["Date"])
news["event_date"] = news["Date"].dt.date 

news_t = news[news["Stock_symbol"] == TARGET_TICKER].copy()

# the lists of date where our target ticker occurs
events_small = (
    news_t
    .drop_duplicates(subset=["Stock_symbol", "event_date"])
    .loc[:, ["Stock_symbol", "event_date"]]
    .rename(columns={"Stock_symbol": "ticker"})
    .sort_values(["ticker", "event_date"])
    .reset_index(drop=True)
)

events_small.head(), len(events_small)

(  ticker  event_date
 0    AAL  2020-11-23
 1    AAL  2020-11-24
 2    AAL  2020-11-25
 3    AAL  2020-11-26
 4    AAL  2020-11-27,
 809)

3. Stock prices

In [32]:
PRICE_DIR = DATA_RAW / "full_history"
price_path = PRICE_DIR / "AAL.csv"

prices_small = pd.read_csv(price_path)
print(prices_small.columns)

prices_small["date"] = pd.to_datetime(prices_small["date"]).dt.date
prices_small = prices_small.sort_values("date").reset_index(drop=True)
prices_small.head(), prices_small.tail()

Index(['date', 'open', 'high', 'low', 'close', 'adj close', 'volume'], dtype='object')


(         date       open       high        low      close  adj close   volume
 0  2005-09-27  21.049999  21.400000  19.100000  19.299999  18.194910   961200
 1  2005-09-28  19.299999  20.530001  19.200001  20.500000  19.326199  5747900
 2  2005-09-29  20.400000  20.580000  20.100000  20.209999  19.052801  1078200
 3  2005-09-30  20.260000  21.049999  20.180000  21.010000  19.806999  3123300
 4  2005-10-03  20.900000  21.750000  20.900000  21.500000  20.268938  1057900,
             date   open   high    low  close  adj close    volume
 4590  2023-12-21  14.21  14.43  14.20  14.35      14.35  30372600
 4591  2023-12-22  14.38  14.40  14.21  14.31      14.31  25169900
 4592  2023-12-26  14.25  14.26  14.04  14.11      14.11  22157900
 4593  2023-12-27  14.10  14.18  13.91  13.99      13.99  23428500
 4594  2023-12-28  13.92  14.04  13.82  13.98      13.98  17069900)

4. Calculate returns

In [40]:
prices_small["ret"] = prices_small["close"].pct_change()
prices_small["trading_idx"] = np.arange(len(prices_small))
idx_by_date = dict(zip(prices_small["date"], prices_small["trading_idx"]))
prices_small.head()

,date,open,high,low,close,adj close,volume,ret,trading_idx
0,2005-09-27,21.049999,21.400000,19.100000,19.299999,18.194910,961200,NaN,0
1,2005-09-28,19.299999,20.530001,19.200001,20.500000,19.326199,5747900,0.062176,1
2,2005-09-29,20.400000,20.580000,20.100000,20.209999,19.052801,1078200,-0.014146,2
3,2005-09-30,20.260000,21.049999,20.180000,21.010000,19.806999,3123300,0.039584,3
4,2005-10-03,20.900000,21.750000,20.900000,21.500000,20.268938,1057900,0.023322,4


5. Calculate CAR

In [41]:
def car_m1p3(event_date, prices_df, idx_map):
    if event_date not in idx_map:
        return np.nan
    
    center = idx_map[event_date]
    start_idx = center - 1
    end_idx = center + 3

    mask = (prices_df["trading_idx"] >= start_idx) & (prices_df["trading_idx"] <= end_idx)
    window = prices_df[mask]

    return window["ret"].sum()

def add_car_to_events(events_df, prices_df, idx_map):
    cars = []
    for d in events_df["event_date"]:
        car = car_m1p3(d, prices_df, idx_map)
        cars.append(car)
    out = events_df.copy()
    out["car_m1p3"] = cars
    return out

events_with_car = add_car_to_events(events_small, prices_small, idx_by_date)
events_with_car.head()

,ticker,event_date,car_m1p3
0,AAL,2020-11-23,0.165569
1,AAL,2020-11-24,0.129155
2,AAL,2020-11-25,0.056861
3,AAL,2020-11-26,NaN
4,AAL,2020-11-27,0.005286
